# Analyze: Multiple Scenario PartMC vs. MAM4

This notebook demonstrates how to analyze results from a multiple-scenario AMBRS comparison of PartMC and MAM4. You will use a variety of diagnostic plots to understand differences in structural errors between the PartMC single-particle model and the MAM4 modal model.

By the end of this tutorial, you should be able to:
* Understand AMBRS "cases" and how they are applied to various aerosol models
* Plot diagnostic properties from AMBRS output data
* Use diagnostics to understand the effects of structural error in PartMC and MAM4
* Apply this understanding of structural error effects to predict differences in additional cases

<small>__Note:__ This notebook expects AMBRS output data for the multiple-scenario comparison of PartMC and MAM4 to exist in `tutorial/data/prepared/multiple_scenario_partmc/`. The output data can be regenerated using `tutorial/notebooks/01_run_models/02_multiple_scenario_partmc.ipynb`.</small>

## About our Models

PartMC:

MAM4:

## Setup

Here we'll pull in AMBRS dependencies and create a few helpers so that it's easy to navigate the AMBRS data.

In [ ]:
import pickle
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display
from part2pop.viz.style import StyleManager, Theme
from part2pop.viz.builder import build_plotter

tutorial_root = Path.cwd().parent.parent

# post-processed diagnostic data will be saved here
prepared_output_dir = (
    tutorial_root
    / 'data'
    / 'prepared'
    / 'multiple_scenario_partmc_mam4'
)

process_dropdown = widgets.Dropdown(
    options=[
        'coagulation only',
        'condensation only',
        'condensation with coagulation',
    ],
    value='coagulation only',
    description='Select processes:',
    disabled=False
)

scenario_dropdown = widgets.Dropdown(
    options=[
        'base case',
        'high particle count',
        'high sulfuric acid',
    ],
    value='base case',
    description='Select a scenario:',
    disabled=False,
)

diagnostic_dropdown = widgets.Dropdown(
    options=[
        'b_abs',
        'dNdlnD',
        'geom_mean_diameter',
        'geom_std_dev',
        'Nccn',
        'number_concentration',
    ]
)

is_snapshot = ['b_abs', 'dNdlnD', 'Nccn']
timestep_dropdown = widgets.Dropdown(
    options=[
        '0',
        '12', 
        '24',
        '48',
        '59',
    ]
)


## AMBRS Cases

In AMBRS, "cases" are used to create consistent sets of conditions for aerosol box models. These conditions include, e.g., the temperature and pressure of the air mass, the initial number, size, and composition of aerosol particles, and the concentrations of gas-phase chemical species.

Let's select a case to investigate:

In [ ]:
display(scenario_dropdown)


Using consistent conditions (and multiple cases) facilitates investigation of differences in structural errors among models. These errors begin to have an effect even before the simulations begin! This is because each model's representation of an aerosol system bakes in assumptions that limit its ability to accurately capture every detail about a real aerosol system.

We'll take a look at the differences in how PartMC and MAM4 represent the initial aerosol systems in our selected case using diagnostics from AMBRS. But first, try to predict what differences you would expect based on the assumptions made in PartMC and MAM4. Then plot the diagnostic and see if you were correct:

In [ ]:
display(process_dropdown)

In some cases, the four modes used by MAM4 to represent an aerosol system limit its ability to accurately capture certain aerosol populations, leading to differences in the initial size distribution compared to the higher-resolution PartMC model.

_What types of aerosol systems would PartMC's assumptions prevent it from accurately capturing?_

## Predict the Differences

Now, let's look at comparisons of full simulations using PartMC and MAM4. Choose a diagnostic to plot as a function of time:

In [ ]:
display(diagnostic_dropdown)

Some diagnostics are plotted as time series (number concentration, GMD, and GSD). The others (number of CCN, total absorption, dN/dln(D)) can be plotted at specific timesteps.

In [ ]:
if diagnostic_dropdown.value in is_snapshot:
    display(timestep_dropdown)

_Based on what you know of how PartMC and MAM4 represent aerosol systems, what differences would you expect to see in this property over time?_

## Plot the Results

Let's see if you were right! Plot the chosen diagnostic over time:

In [ ]:
suffix = (f'-{timestep_dropdown.value}' if diagnostic_dropdown.value in is_snapshot else '')
pkl_file = f'{process_dropdown.value}-{scenario_dropdown.value}-{diagnostic_dropdown.value}{suffix}.pkl'.replace(' ', '-')
with (prepared_output_dir / pkl_file).open('rb') as f:
    payload = pickle.load(f)

mgr = StyleManager(Theme(), deterministic=False)
plotter = build_plotter(
    payload['plotter_type'],
    payload['plotter_config'],
)

series = [
    {"key": "partmc", "population": payload['prepared']['partmc'], "label": "PartMC-MOSAIC"},
    {"key": "mam4", "population": payload['prepared']['mam4'], "label": "MAM4"},
]
line_styles = mgr.plan("line", [s["key"] for s in series])

fig, ax = plt.subplots()
for s in series:
    cfg = {
        "varname": diagnostic_dropdown.value,
        "style": line_styles[s["key"]],
    }
    plotter = build_plotter(payload['plotter_type'], cfg)
    plotter.plot(ax=ax, prepared=s['population'], label=s["label"])
time_text = (f'; time = {payload["metadata"]["time"]} s' if 'time' in payload['metadata'] else '')
fig.text(0, 0, f'{process_dropdown.value} - {scenario_dropdown.value}{time_text}')
ax.legend()
fig.tight_layout()